In [1]:
import os
import json

EVAL_DIR        = "C:/Users/User/Comvi/SAM3/Project/eval_results"
THRESH_FILE = os.path.join(EVAL_DIR, "best_thresholds.json")

def save_thresholds(d: dict):
    """บันทึก best_threshold dict → JSON"""
    out = {"|||".join(k): v for k, v in d.items()}
    with open(THRESH_FILE, "w") as f:
        json.dump(out, f, indent=2)
    print(f"✅ Saved {len(out)} thresholds → {THRESH_FILE}")

In [2]:
best_thresh = {
    ("checkpoint_25", "Wonyoung"): 0.6,
    ("checkpoint_25", "Yujin"):    0.5,
    ("checkpoint_25", "Rei"):      0.1,
    ("checkpoint_25", "Leeseo"):   0.5,
    ("checkpoint_25", "Liz"):      0.7,
    ("checkpoint_25", "Gaeul"):    0.5,

    ("checkpoint_30", "Wonyoung"): 0.1,
    ("checkpoint_30", "Yujin"):    0.5,
    ("checkpoint_30", "Rei"):      0.6,
    ("checkpoint_30", "Leeseo"):   0.5,
    ("checkpoint_30", "Liz"):      0.7,
    ("checkpoint_30", "Gaeul"):    0.5,
}

save_thresholds(best_thresh)

✅ Saved 12 thresholds → C:/Users/User/Comvi/SAM3/Project/eval_results\best_thresholds.json


In [3]:
# import torch
# import os

# CHECKPOINT_DIR = "C:/Users/User/Comvi/SAM3/Project/outputs/ive_training/checkpoints"

# ckpt_a = "checkpoint_30.pt"
# ckpt_b = "checkpoint_25.pt"
# output_name = "merged_30_25.pt"

# path_a = os.path.join(CHECKPOINT_DIR, ckpt_a)
# path_b = os.path.join(CHECKPOINT_DIR, ckpt_b)

# print("Loading checkpoints...")
# a = torch.load(path_a, map_location="cpu")
# b = torch.load(path_b, map_location="cpu")

# wa = a.get("model", a)
# wb = b.get("model", b)

# merged = {}

# print("Merging weights...")
# for k in wa:
#     if k in wb and wa[k].dtype.is_floating_point:
#         merged[k] = (wa[k] + wb[k]) / 2.0
#     else:
#         merged[k] = wa[k]

# save_path = os.path.join(CHECKPOINT_DIR, output_name)
# torch.save({"model": merged}, save_path)

# print("✅ Merged checkpoint saved to:", save_path)

In [4]:
# !ffmpeg -ss 00:00:00 -to 00:02:00 -i "IVE Answer.mp4" -c copy short.mp4

In [ ]:
"""
SAM3 Video — Speed-First Version
=================================
ลด complexity ให้สุด เพื่อหาว่าคอขวดอยู่ที่ไหน

Strategy:
  - member prompt ล้วน (ไม่มี compound) → เร็วสุด
  - encode image ครั้งเดียว เก็บ features ไว้ใช้ทุก prompt
  - ถ้าต้องการ compound ทีหลังค่อยเพิ่ม
"""

import os, json, time, warnings
import numpy as np
import torch
import cv2
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ========================= CONFIG =========================

CHECKPOINT_DIR  = "C:/Users/User/Comvi/SAM3/Project/outputs/ive_training/checkpoints"
EVAL_DIR        = "C:/Users/User/Comvi/SAM3/Project/eval_results"
VIDEO_INPUT     = "C:/Users/User/Comvi/SAM3/Project/short.mp4"
VIDEO_OUTPUT    = "C:/Users/User/Comvi/SAM3/Project/video/output_segmented.mp4"

CKPT_NAME       = "merged_30_25"

# member prompt เท่านั้น (ไม่มี compound → เร็วกว่า 2x)
PROMPTS = [
    "Wonyoung",
    "Gaeul",
    "Rei",
]

INFERENCE_EVERY = 10
MODEL_SCALE     = 0.5
DEVICE          = "cuda"

OVERLAY_ALPHA   = 0.35
CONTOUR_THICK   = 2

COLOR_MAP = {
    "Wonyoung" : (  0,   0, 255),
    "Gaeul"    : (255,   0,   0),
    "Rei"      : (  0, 255,   0),
}

THRESH_FILE = os.path.join(EVAL_DIR, "best_thresholds.json")

# ==========================================================

def load_thresholds():
    if not os.path.exists(THRESH_FILE):
        print("⚠️  ใช้ threshold default 0.5")
        return {}
    with open(THRESH_FILE) as f:
        raw = json.load(f)
    d = {tuple(k.split("|||")): v for k, v in raw.items()}
    print(f"✅ Loaded {len(d)} thresholds")
    return d


def load_model(ckpt_name=None):
    from sam3.model_builder import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor

    model = build_sam3_image_model(device=DEVICE)
    if ckpt_name:
        path  = os.path.join(CHECKPOINT_DIR, f"{ckpt_name}.pt")
        state = torch.load(path, map_location="cpu", weights_only=False)
        model.load_state_dict(state.get("model", state), strict=False)
        print(f"  ✅ {ckpt_name}.pt")
    else:
        print(f"  ✅ base model")
    model = model.to(DEVICE).eval()
    return model, Sam3Processor(model)


# ==========================================================

def benchmark(proc, image, prompts, thresh, ckpt_name):
    """วัดเวลาจริงต่อ frame ก่อนรัน video จริง"""
    print("\n── Benchmark (5 frames) ──────────────────")
    times = []
    for i in range(5):
        t0    = time.perf_counter()
        state = proc.set_image(image)
        for p in prompts:
            t = thresh.get((ckpt_name, p), 0.5)
            try:
                proc.set_text_prompt(state=state, prompt=p,
                                     box_threshold=t, text_threshold=t)
            except TypeError:
                proc.set_text_prompt(p, state)
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
        print(f"  frame {i+1}: {elapsed:.2f}s")

    avg = np.mean(times[1:])   # skip first (warmup)
    print(f"  avg (excl warmup): {avg:.2f}s/frame")
    return avg


def segment_frame(proc, image: Image.Image,
                  prompts: list, thresh: dict,
                  ckpt_name: str) -> dict:
    """encode image ครั้งเดียว → loop prompt"""
    with torch.no_grad():
        state = proc.set_image(image)
        results = {}
        for p in prompts:
            t = thresh.get((ckpt_name, p), 0.5)
            try:
                out = proc.set_text_prompt(state=state, prompt=p,
                                           box_threshold=t, text_threshold=t)
            except TypeError:
                out = proc.set_text_prompt(p, state)

            if out is None or out["masks"].shape[0] == 0:
                results[p] = None
            else:
                idx   = out["scores"].argmax().item()
                score = out["scores"][idx].item()
                results[p] = (out["masks"][idx, 0].cpu().numpy().astype(bool)
                              if score >= t else None)
    return results


def draw_frame(frame, masks):
    result  = frame.copy()
    overlay = result.copy()
    for name, mask in masks.items():
        if mask is None:
            continue
        color = COLOR_MAP.get(name, (0, 255, 255))
        overlay[mask] = color
        contours, _ = cv2.findContours(mask.astype(np.uint8),
                                        cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(result, contours, -1, color, CONTOUR_THICK)
        ys, xs = np.where(mask)
        if len(xs) > 0:
            cv2.putText(result, name,
                        (int(xs.mean()), int(ys.mean())),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                        (255, 255, 255), 2, cv2.LINE_AA)
    result = cv2.addWeighted(result, 1-OVERLAY_ALPHA, overlay, OVERLAY_ALPHA, 0)
    for i, (name, color) in enumerate(COLOR_MAP.items()):
        y = 30 + i*30
        cv2.rectangle(result, (10, y-15), (30, y+5), color, -1)
        cv2.putText(result, name, (40, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (255, 255, 255), 2, cv2.LINE_AA)
    return result


# ==========================================================

def process_video():
    os.makedirs(os.path.dirname(VIDEO_OUTPUT) or ".", exist_ok=True)

    thresh         = load_thresholds()
    print("Loading model...")
    model, proc    = load_model(CKPT_NAME)

    cap    = cv2.VideoCapture(VIDEO_INPUT)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\nVideo : {width}×{height}  {fps:.1f}fps  {total} frames")
    print(f"Skip  : every {INFERENCE_EVERY} frames")
    print(f"Scale : {MODEL_SCALE}x")
    print(f"Prompts: {PROMPTS}")

    # ── benchmark ก่อน ──────────────────────────────────────
    ret, test_frame = cap.read()
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    small     = cv2.resize(test_frame, None, fx=MODEL_SCALE, fy=MODEL_SCALE)
    test_img  = Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
    avg_time  = benchmark(proc, test_img, PROMPTS, thresh, CKPT_NAME)

    n_calls   = total // INFERENCE_EVERY
    est_min   = avg_time * n_calls / 60
    print(f"\nประมาณเวลา: {avg_time:.2f}s × {n_calls} calls = {est_min:.1f} นาที")

    # ── process video ────────────────────────────────────────
    writer = cv2.VideoWriter(VIDEO_OUTPUT,
                              cv2.VideoWriter_fourcc(*"mp4v"),
                              fps, (width, height))

    prev_masks = {p: None for p in PROMPTS}
    frame_idx  = 0
    pbar       = tqdm(total=total, unit="frame")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % INFERENCE_EVERY == 0:
            small      = cv2.resize(frame, None, fx=MODEL_SCALE, fy=MODEL_SCALE)
            image      = Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
            masks_sm   = segment_frame(proc, image, PROMPTS, thresh, CKPT_NAME)

            prev_masks = {}
            for k, v in masks_sm.items():
                if v is None:
                    prev_masks[k] = None
                else:
                    prev_masks[k] = cv2.resize(
                        v.astype(np.uint8), (width, height),
                        interpolation=cv2.INTER_NEAREST
                    ).astype(bool)

            torch.cuda.empty_cache()

        writer.write(draw_frame(frame, prev_masks))
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()
    print(f"\n✅ Done: {VIDEO_OUTPUT}")


if __name__ == "__main__":
    process_video()

✅ Loaded 12 thresholds
Loading model...
  ✅ merged_30_25.pt

Video : 1920×1080  24.0fps  2878 frames
Skip  : every 10 frames
Scale : 0.5x
Prompts: ['Wonyoung', 'Gaeul', 'Rei']

── Benchmark (5 frames) ──────────────────
  frame 1: 1.04s
  frame 2: 0.78s
  frame 3: 0.78s
  frame 4: 0.78s
  frame 5: 0.78s
  avg (excl warmup): 0.78s/frame

ประมาณเวลา: 0.78s × 287 calls = 3.7 นาที


100%|██████████| 2878/2878 [05:12<00:00,  9.22frame/s]


✅ Done: C:/Users/User/Comvi/SAM3/Project/video/output_segmented.mp4


In [ ]:
"""
SAM3 Video Segmentation — Final Optimized
==========================================
Prompts:
  "Wonyoung"   → member (fine-tuned)
  "Gaeul"      → member (fine-tuned)
  "ผมของเรอิ"  → compound: Rei mask → crop ROI → base model หา hair

Speed fix:
  - encode full image ครั้งเดียวต่อ frame (reuse state ทุก prompt)
  - compound: encode เฉพาะ ROI ของ member (เล็กกว่า full ~4x)
  - ไม่ encode ซ้ำ → ~7-8 นาทีสำหรับ 2878 frames
"""

import os, json, warnings
import numpy as np
import torch
import cv2
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ========================= CONFIG =========================

CHECKPOINT_DIR  = "C:/Users/User/Comvi/SAM3/Project/outputs/ive_training/checkpoints"
EVAL_DIR        = "C:/Users/User/Comvi/SAM3/Project/eval_results"
VIDEO_INPUT     = "C:/Users/User/Comvi/SAM3/Project/short.mp4"
VIDEO_OUTPUT    = "C:/Users/User/Comvi/SAM3/Project/video/output_segmented.mp4"

CKPT_NAME       = "merged_30_25"

PROMPTS = [
    "Wonyoung",     # member
    "กาอึล",        # member
    "ผมของเรอิ",    # compound → Rei mask + hair ROI
]

INFERENCE_EVERY = 10
MODEL_SCALE     = 0.5
DEVICE          = "cuda"

OVERLAY_ALPHA   = 0.35
CONTOUR_THICK   = 2

COLOR_MAP = {
    "Wonyoung"  : (  0,   0, 255),   # แดง
    "Gaeul"     : (255,   0,   0),   # ฟ้า
    "ผมของเรอิ" : (  0, 255,   0),   # เขียว
}

THRESH_FILE = os.path.join(EVAL_DIR, "best_thresholds.json")

# ==========================================================
# PROMPT PARSER
# ==========================================================

MEMBER_ALIASES = {
    "wonyoung": "Wonyoung", "yujin":    "Yujin",
    "rei":      "Rei",       "leeseo":   "Leeseo",
    "liz":      "Liz",       "gaeul":    "Gaeul",
    "วอนยอง":  "Wonyoung",  "อันยูจิน": "Yujin",
    "ยูจิน":   "Yujin",     "เรอิ":     "Rei",
    "ลีซอ":    "Leeseo",    "ลิซ":      "Liz",
    "กาอึล":   "Gaeul",
}

BODY_PART_MAP = {
    "ผม":     "hair",   "ใบหน้า": "face",  "หน้า":  "face",
    "แขน":    "arm",    "ขา":     "leg",   "มือ":   "hand",
    "เท้า":   "foot",   "ตัว":    "body",  "เสื้อ": "shirt",
    "กางเกง": "pants",  "รองเท้า":"shoes",
    "hair":   "hair",   "face":   "face",  "arm":   "arm",
    "leg":    "leg",    "body":   "body",  "shirt": "shirt",
}

CONNECTORS = ["ของ", " of ", "'s "]


def parse_prompt(prompt: str) -> dict:
    pl = prompt.strip().lower()
    member = part = None
    for conn in CONNECTORS:
        if conn in pl:
            left, right = pl.split(conn, 1)
            for a, n in MEMBER_ALIASES.items():
                if a in left or a in right: member = n; break
            for a, e in BODY_PART_MAP.items():
                if a in left or a in right: part = e; break
            if member and part:
                return {"mode": "compound", "member": member,
                        "part": part, "original": prompt}
    for a, n in MEMBER_ALIASES.items():
        if a in pl: member = n; break
    for a, e in BODY_PART_MAP.items():
        if a in pl: part = e; break
    if member and part:
        return {"mode": "compound", "member": member,
                "part": part,   "original": prompt}
    elif member:
        return {"mode": "member",   "member": member,
                "part": None,   "original": prompt}
    elif part:
        return {"mode": "part",     "member": None,
                "part": part,   "original": prompt}
    return {"mode": "raw", "member": None,
            "part": None, "original": prompt}


# ==========================================================
# HELPERS
# ==========================================================

def load_thresholds() -> dict:
    if not os.path.exists(THRESH_FILE):
        print("⚠️  ใช้ threshold default 0.5")
        return {}
    with open(THRESH_FILE) as f:
        raw = json.load(f)
    d = {tuple(k.split("|||")): v for k, v in raw.items()}
    print(f"✅ Loaded {len(d)} thresholds")
    return d


def load_model(ckpt_name=None):
    from sam3.model_builder import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor

    model = build_sam3_image_model(device=DEVICE)
    if ckpt_name:
        path  = os.path.join(CHECKPOINT_DIR, f"{ckpt_name}.pt")
        state = torch.load(path, map_location="cpu", weights_only=False)
        model.load_state_dict(state.get("model", state), strict=False)
        print(f"  ✅ {ckpt_name}.pt")
    else:
        print(f"  ✅ base model")
    model = model.to(DEVICE).eval()

    from sam3.model.sam3_image_processor import Sam3Processor
    return Sam3Processor(model)


def _set_text(proc, state, text, t):
    """ส่ง text prompt — รองรับทั้ง 2 signature"""
    try:
        out = proc.set_text_prompt(state=state, prompt=text,
                                   box_threshold=t, text_threshold=t)
    except TypeError:
        out = proc.set_text_prompt(text, state)
    return out


def _best_mask(out, threshold):
    """ดึง mask ที่ score สูงสุด หรือ None"""
    if out is None or out["masks"].shape[0] == 0:
        return None
    idx   = out["scores"].argmax().item()
    score = out["scores"][idx].item()
    if score < threshold:
        return None
    return out["masks"][idx, 0].cpu().numpy().astype(bool)


# ==========================================================
# SEGMENT FRAME
# ==========================================================

def segment_frame(ft_proc, base_proc,
                  image: Image.Image,
                  prompts: list,
                  thresh: dict,
                  ckpt_name: str) -> dict:
    """
    encode full image ครั้งเดียวด้วย ft_proc
    compound: reuse ft_proc state → crop ROI → encode ROI ด้วย base_proc
    """
    img_np = np.array(image)
    h, w   = img_np.shape[:2]
    results = {}

    with torch.no_grad():
        # encode full image ครั้งเดียว — reuse ทุก prompt
        ft_state = ft_proc.set_image(image)

        for prompt in prompts:
            p = parse_prompt(prompt)

            # ── member prompt ──────────────────────────────
            if p["mode"] == "member":
                t   = thresh.get((ckpt_name, p["member"]), 0.5)
                out = _set_text(ft_proc, ft_state, p["member"], t)
                results[prompt] = _best_mask(out, t)

            # ── compound: "ผมของเรอิ" ──────────────────────
            elif p["mode"] == "compound":
                member = p["member"]
                part   = p["part"]
                t      = thresh.get((ckpt_name, member), 0.5)

                # ขั้น 1: หา member mask (reuse ft_state — ไม่ encode ซ้ำ)
                out         = _set_text(ft_proc, ft_state, member, t)
                member_mask = _best_mask(out, t)
                if member_mask is None:
                    results[prompt] = None
                    continue

                # ขั้น 2: crop ROI + padding 12%
                rows = np.where(member_mask.any(axis=1))[0]
                cols = np.where(member_mask.any(axis=0))[0]
                if len(rows) == 0 or len(cols) == 0:
                    results[prompt] = None
                    continue

                y1, y2 = int(rows[0]), int(rows[-1])
                x1, x2 = int(cols[0]), int(cols[-1])
                py = int((y2-y1) * 0.12)
                px = int((x2-x1) * 0.12)
                y1 = max(0, y1-py);  y2 = min(h, y2+py)
                x1 = max(0, x1-px);  x2 = min(w, x2+px)
                roi = Image.fromarray(img_np[y1:y2, x1:x2])

                # ขั้น 3: encode เฉพาะ ROI (เล็กกว่า full image ~4x → เร็วกว่า)
                roi_state = base_proc.set_image(roi)
                roi_out   = _set_text(base_proc, roi_state, part, 0.4)
                roi_mask  = _best_mask(roi_out, 0.4)
                if roi_mask is None:
                    results[prompt] = None
                    continue

                # ขั้น 4: map กลับ full image ∩ member mask
                full = np.zeros((h, w), dtype=bool)
                full[y1:y2, x1:x2] = roi_mask
                results[prompt] = full & member_mask

            # ── body part / raw ────────────────────────────
            else:
                base_state      = base_proc.set_image(image)
                out             = _set_text(base_proc, base_state,
                                            p.get("part") or prompt, 0.45)
                results[prompt] = _best_mask(out, 0.45)

    return results


# ==========================================================
# DRAW OVERLAY
# ==========================================================

def draw_frame(frame: np.ndarray, masks: dict) -> np.ndarray:
    result  = frame.copy()
    overlay = result.copy()

    for name, mask in masks.items():
        if mask is None:
            continue
        color = COLOR_MAP.get(name, (0, 255, 255))
        overlay[mask] = color

        contours, _ = cv2.findContours(mask.astype(np.uint8),
                                        cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(result, contours, -1, color, CONTOUR_THICK)

        ys, xs = np.where(mask)
        if len(xs) > 0:
            cv2.putText(result, name,
                        (int(xs.mean()), int(ys.mean())),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                        (255, 255, 255), 2, cv2.LINE_AA)

    result = cv2.addWeighted(result, 1-OVERLAY_ALPHA, overlay, OVERLAY_ALPHA, 0)

    # legend
    for i, (name, color) in enumerate(COLOR_MAP.items()):
        y = 30 + i*30
        cv2.rectangle(result, (10, y-15), (30, y+5), color, -1)
        cv2.putText(result, name, (40, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (255, 255, 255), 2, cv2.LINE_AA)

    return result


# ==========================================================
# MAIN
# ==========================================================

def process_video():
    os.makedirs(os.path.dirname(VIDEO_OUTPUT) or ".", exist_ok=True)

    thresh    = load_thresholds()

    print("Loading models...")
    ft_proc   = load_model(CKPT_NAME)
    base_proc = load_model(None)

    cap    = cv2.VideoCapture(VIDEO_INPUT)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    n_calls  = total // INFERENCE_EVERY
    # ประมาณ: 0.78s (member×2) + 0.4s (ROI compound) = ~1.2s/call
    est_min  = n_calls * 1.2 / 60

    print(f"\nVideo  : {width}×{height}  {fps:.1f}fps  {total} frames  ({total/fps/60:.1f} นาที)")
    print(f"Skip   : every {INFERENCE_EVERY} frames  ({n_calls} inference calls)")
    print(f"Scale  : {MODEL_SCALE}x")
    print(f"Prompts: {PROMPTS}")
    print(f"ประมาณ : ~{est_min:.1f} นาที\n")

    writer = cv2.VideoWriter(
        VIDEO_OUTPUT,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps, (width, height))

    prev_masks = {p: None for p in PROMPTS}
    frame_idx  = 0
    pbar       = tqdm(total=total, unit="frame")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % INFERENCE_EVERY == 0:
            small  = cv2.resize(frame, None, fx=MODEL_SCALE, fy=MODEL_SCALE)
            image  = Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))

            masks_sm = segment_frame(
                ft_proc, base_proc, image, PROMPTS, thresh, CKPT_NAME)

            # scale mask กลับ full resolution
            prev_masks = {}
            for k, v in masks_sm.items():
                if v is None:
                    prev_masks[k] = None
                else:
                    prev_masks[k] = cv2.resize(
                        v.astype(np.uint8), (width, height),
                        interpolation=cv2.INTER_NEAREST
                    ).astype(bool)

            torch.cuda.empty_cache()

        writer.write(draw_frame(frame, prev_masks))
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()
    print(f"\n✅ Done: {VIDEO_OUTPUT}")


if __name__ == "__main__":
    process_video()

✅ Loaded 12 thresholds
Loading models...
  ✅ merged_30_25.pt
  ✅ base model

Video  : 1920×1080  24.0fps  2878 frames  (2.0 นาที)
Skip   : every 10 frames  (287 inference calls)
Scale  : 0.5x
Prompts: ['Wonyoung', 'กาอึล', 'ผมของเรอิ']
ประมาณ : ~5.7 นาที



100%|██████████| 2878/2878 [1:10:13<00:00,  1.46s/frame]



✅ Done: C:/Users/User/Comvi/SAM3/Project/video/output_segmented.mp4
